# Baomai R43 — all rows of one frame together

**Use a standard CPU runtime and run all cells.** This opens the cumulative review and downloads its portable ZIP. No GPU, training, model inference, API key or account token is needed.

- **Previous/Next switches frames.** All rows of the selected photo are visible together by default, including unchanged rows.
- Each row shows original pixels, the working baseline, previous R42 and latest R43, with exact text and review history.
- **10 requested rows reviewed: 9 amended, 1 retained.** Nine corner slots and one right box boundary change. `frame_0160` source row 31 retains its current left point because the pixels do not establish a better replacement.
- All **508 identities** remain across 12 frames, including the earlier deleted identity. There are **507 current fronts**.
- Three complete assistant-reviewed candidates remain: **frame_0146, frame_0371, frame_0383**. Nine whole frames and the same 55 source rows remain held.

The `frame_*` originals come from an existing additional 320-image upstream collection, separate from the 157 `new_*`/`old_*` working photos. The review embeds the actual 12 selected original JPEGs. The self-contained `review.html` also works offline after download.


In [ ]:
from pathlib import Path
import base64, hashlib, html, json, subprocess, sys, tempfile, urllib.request, zipfile

# Immutable Git object and content hashes. A branch moving cannot substitute a different capsule.
REPOSITORY = "Siuuuers/dl"
ARCHIVE_BLOB = "ab2fc7fbd37801b8a6b3cf1ce21002aa16f5b408"
ARCHIVE_SHA256 = "ae3d7310fd6b2d3a623bd7b8760dc9d6ffc75e6df1be28bbf0a2555fdb380d6e"
ARCHIVE_BYTES = 4293004
MANIFEST_SHA256 = "d40787277c0da7487ee8712da659be847ee9c972918b5cee420488dbafdc499f"
LOCAL_REVIEW_ARCHIVE = ""  # Optional: an already downloaded R43 review.zip, for offline use.
DOWNLOAD_BUNDLE = True

work = Path(tempfile.mkdtemp(prefix="baomai_r43_"))
archive_path = work / "baomai_front_review_r43.zip"
if LOCAL_REVIEW_ARCHIVE:
    content = Path(LOCAL_REVIEW_ARCHIVE).read_bytes()
else:
    url = f"https://api.github.com/repos/{REPOSITORY}/git/blobs/{ARCHIVE_BLOB}"
    request = urllib.request.Request(url, headers={"Accept":"application/vnd.github+json", "User-Agent":"baomai-r43-review"})
    with urllib.request.urlopen(request, timeout=90) as response:
        result = json.load(response)
    assert result["sha"] == ARCHIVE_BLOB and result["encoding"] == "base64", "Unexpected Git object"
    content = base64.b64decode(result["content"])
assert len(content) == ARCHIVE_BYTES and hashlib.sha256(content).hexdigest() == ARCHIVE_SHA256, "Capsule identity mismatch"
archive_path.write_bytes(content)
folder = work / "review"
with zipfile.ZipFile(archive_path) as z:
    items = z.infolist()
    names = [x.filename for x in items]
    assert len(names) == len(set(names)), "Duplicate ZIP member"
    assert sum(x.file_size for x in items) < 100_000_000, "Unexpected expanded size"
    assert z.testzip() is None, "CRC failure"
    inventory = json.loads(z.read("inventory.json"))
    assert set(names) == set(inventory) | {"inventory.json"}, "Inventory mismatch"
    for item in items:
        rel = Path(item.filename)
        assert not rel.is_absolute() and ".." not in rel.parts and "\\" not in item.filename
        assert not item.is_dir() and ((item.external_attr >> 16) & 0o170000) != 0o120000
        payload = z.read(item.filename)
        if item.filename in inventory:
            assert inventory[item.filename] == {"sha256":hashlib.sha256(payload).hexdigest(), "bytes":len(payload)}, item.filename
        target = folder / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
assert hashlib.sha256((folder / "manifest.json").read_bytes()).hexdigest() == MANIFEST_SHA256
checked = subprocess.run([sys.executable, str(folder / "review.py"), "--check-capsule", str(archive_path)], check=True, capture_output=True, text=True)
verification = json.loads(checked.stdout)
print("Verified all 12 original images, 508 identities and 507 current fronts.")
print("This round:", verification["latest_counts"]["amended_rows"], "amended rows;", verification["latest_counts"]["retained_rows"], "reviewed and retained.")
print("Complete annotation candidates:", ", ".join(verification["annotation_complete_ids"]))
print("Held whole images:", ", ".join(verification["held_image_ids"]))
print("No training or model promotion occurred.")
print("Capsule:", archive_path)


In [ ]:
from IPython.display import HTML, display
viewer = (folder / "review.html").read_text()
display(HTML('<iframe title="R43 original-pixel review" sandbox="allow-scripts" style="width:100%;height:1100px;border:0" srcdoc="' + html.escape(viewer, quote=True) + '"></iframe>'))
if DOWNLOAD_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        print("Open the capsule path printed above; review.html is self-contained after extraction.")
    else:
        files.download(str(archive_path))


## Reading the latest review

Start with **All rows**, select a frame, and scroll through its cards. The row selector jumps within the same page. The whole-frame image can show upstream, working, R42 or latest R43 labels, or no overlay.

**Magenta means a change relative to the previous displayed review stage:** R42 compares with the working baseline; latest R43 compares only with R42. Open a card's history for exact annotation text and reasons. Older corrections remain visible even if that row did not change this round.

The five saved prediction figures are preserved as **historical R42 comparisons**. They have not been rescored with the latest R43 corners. This notebook does not demonstrate an improved model or reduced unmatched predictions.

For `frame_0001` source row 43, the right border is **xmax**, reduced to nominal 859 px; the right lower corner is nominal (856,657). Its owned upper rim reaches farther right than the lower corner. A tight rectangular box can contain some rack or gripper pixels internally, but those materials alone must not set its outer boundary. The hidden left corner stays `0 0 0`, and the existing left-boundary hold remains.

Soft physical corners support nominal locations, not subpixel certainty. The complete images are still candidate annotations with no assigned training split or human sign-off; the mined pool is not an independent test set.
